# 04 — Data Drift Monitoring with Evidently
## Fraud Detection MLOps Platform

In production, incoming data can shift over time (e.g. fraud patterns change,
transaction sizes increase). This is called **data drift** — and it silently
degrades model performance.

This notebook:
1. Simulates a "production batch" of new transactions with drift
2. Generates an Evidently drift report comparing training vs new data
3. Detects which features have drifted
4. Triggers an automatic model retraining if drift is significant
5. Saves drift reports as HTML for the monitoring dashboard

---
### Install & imports

In [1]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "evidently==0.4.30", "-q"])

print("Evidently 0.4.30 installed & Evidently ready")

result = subprocess.run([sys.executable, "-m", "pip", "show", "evidently"],
                       capture_output=True, text=True)
print(result.stdout)

Evidently 0.4.30 installed & Evidently ready
Name: evidently
Version: 0.4.30
Summary: Open-source tools to analyze, monitor, and debug machine learning model in production.
Home-page: https://github.com/evidentlyai/evidently
Author: Emeli Dral
Author-email: emeli.dral@gmail.com
License: UNKNOWN
Location: F:\physical science\4th year\Internship\Task 20\fraud-detection-mlops-platform\venv\Lib\site-packages
Requires: certifi, dynaconf, fsspec, iterative-telemetry, litestar, nltk, numpy, pandas, plotly, pydantic, PyYAML, requests, rich, scikit-learn, scipy, statsmodels, typer, typing-inspect, ujson, urllib3, uvicorn, watchdog
Required-by: 



In [2]:
import pandas as pd
import numpy as np
import json
import os
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# Make sure we are at project root
if Path("notebooks").exists():
    pass  # already at root
else:
    os.chdir("..")

print("Working directory:", os.getcwd())

# Paths
PROCESSED_DIR = Path("data/processed")
MODELS_DIR    = Path("models")
REPORTS_DIR   = Path("monitoring/reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Load feature names
with open(PROCESSED_DIR / "feature_names.json") as f:
    FEATURES = json.load(f)

print(f"Features: {FEATURES}")
print(f"Reports will be saved to: {REPORTS_DIR}")

Working directory: F:\physical science\4th year\Internship\Task 20\fraud-detection-mlops-platform
Features: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_encoded', 'orig_balance_diff', 'dest_balance_diff', 'orig_balance_zero', 'amount_to_balance_ratio', 'hour_of_day']
Reports will be saved to: monitoring\reports


---
## Step 1 — Load Reference Data (Training Set)

The **reference dataset** is what the model was trained on.
Evidently compares new incoming data against this baseline.

In [3]:
# Load training data as the reference (what model learned from)
X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()

# Combine features + target for Evidently
reference_data = X_train.copy()
reference_data["isFraud"] = y_train.values

print(f"Reference dataset: {reference_data.shape}")
print(f"Fraud rate in reference: {reference_data['isFraud'].mean()*100:.3f}%")
reference_data.head()

Reference dataset: (160000, 13)
Fraud rate in reference: 0.296%


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_encoded,orig_balance_diff,dest_balance_diff,orig_balance_zero,amount_to_balance_ratio,hour_of_day,isFraud
0,546,127254.99,2388.0,0.0,217547.76,344802.75,0,2388.0,127254.99,1,10.000000,18,0
1,328,3149.30,105423.0,102273.7,967676.18,970825.48,0,3149.3,3149.30,0,0.029873,16,0
2,612,584651.82,0.0,0.0,1707031.97,2291683.79,1,0.0,584651.82,1,10.000000,12,0
3,190,228906.97,0.0,0.0,3525556.97,3754463.94,0,0.0,228906.97,1,10.000000,22,0
4,253,41260.18,35706.0,0.0,14536522.01,14577782.20,0,35706.0,41260.19,1,1.155553,13,0


---
## Step 2 — Simulate a Production Batch WITH Drift

In real life this would be new transactions coming in.
drift is simulated by:
- **Shifting amounts** — transactions are larger (fraud patterns evolved)
- **Shifting balances** — account sizes grew
- **Adding noise** — natural variation in new data

This mimics what happens after months in production.

In [4]:
np.random.seed(99)

# Take the test set as the base for our "new production batch"
X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

# Simulate drift — apply realistic shifts to key features
production_data = X_test.copy()

# Stronger drift to guarantee threshold is crossed for demo purposes
# Drift 1: Amounts tripled (major fraud pattern shift)
production_data["amount"] = production_data["amount"] * 3.0 + np.random.normal(0, 2000, len(X_test))

# Drift 2: Account balances grew over time
production_data["oldbalanceOrg"]  = production_data["oldbalanceOrg"]  * 2.5
production_data["newbalanceOrig"] = production_data["newbalanceOrig"] * 2.5
production_data["oldbalanceDest"] = production_data["oldbalanceDest"] * 2.0
production_data["newbalanceDest"] = production_data["newbalanceDest"] * 2.0

# Drift 3: Step shifted — simulate data from a later time period
production_data["step"] = production_data["step"] + 400

# Drift 4: Recalculate engineered features consistently
production_data["orig_balance_diff"] = (
    production_data["oldbalanceOrg"] - production_data["newbalanceOrig"]
)
production_data["dest_balance_diff"] = (
    production_data["newbalanceDest"] - production_data["oldbalanceDest"]
)
production_data["amount_to_balance_ratio"] = (
    production_data["amount"] / (production_data["oldbalanceOrg"] + 1e-9)
).clip(upper=10)

production_data["hour_of_day"] = production_data["step"] % 24

# Clip negatives that can appear from noise
production_data["amount"] = production_data["amount"].clip(lower=0)

production_data["isFraud"] = y_test.values

print(f"Production batch: {production_data.shape}")
print()
print("Distribution comparison (stronger drift):")
print(f"  amount      — ref mean: ${reference_data['amount'].mean():>12,.2f}  |  prod mean: ${production_data['amount'].mean():>12,.2f}")
print(f"  step        — ref mean: {reference_data['step'].mean():>8.1f}           |  prod mean: {production_data['step'].mean():>8.1f}")
print(f"  oldbalOrg   — ref mean: ${reference_data['oldbalanceOrg'].mean():>12,.2f}  |  prod mean: ${production_data['oldbalanceOrg'].mean():>12,.2f}")

Production batch: (40000, 13)

Distribution comparison (stronger drift):
  amount      — ref mean: $  316,759.78  |  prod mean: $  956,024.35
  step        — ref mean:    242.0           |  prod mean:    642.0
  oldbalOrg   — ref mean: $   47,352.71  |  prod mean: $  123,232.57


---
## Step 3 — Generate Evidently Drift Report

Evidently compares every feature's distribution between reference and production.
It uses statistical tests (KS test for numerical, chi-squared for categorical)
and flags features where the distributions have shifted significantly.

In [5]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

# Build the drift report
# DataDriftPreset checks every feature column automatically
report = Report(metrics=[
    DataDriftPreset(columns=FEATURES),  # explicitly only feature columns
    DataQualityPreset(),
])

print("Running Evidently drift analysis...")
report.run(
    reference_data = reference_data[FEATURES + ["isFraud"]],
    current_data   = production_data[FEATURES + ["isFraud"]]
)
print("Analysis complete")

# Save as HTML report
report_path = REPORTS_DIR / f"drift_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
report.save_html(str(report_path))
print(f"   Report saved: {report_path}")
print(f"   Open this file in the browser to see the full interactive report")

Running Evidently drift analysis...
Analysis complete
   Report saved: monitoring\reports\drift_report_20260529_123957.html
   Open this file in the browser to see the full interactive report


---
## Step 4 — Extract Drift Results Programmatically

Need to read the drift results in code so It can be decided
whether to trigger retraining automatically.

In [6]:
# First let's see the exact keys available in results
results = report.as_dict()

print("Top-level metrics count:", len(results["metrics"]))
print()

# Check what keys exist in the first metric result
drift_result = results["metrics"][0]["result"]
print("Keys in drift result:")
for key in drift_result.keys():
    print(f"  {key}")

Top-level metrics count: 18

Keys in drift result:
  drift_share
  number_of_columns
  number_of_drifted_columns
  share_of_drifted_columns
  dataset_drift


In [7]:
results = report.as_dict()

# Metric 0 has the overall summary
drift_summary   = results["metrics"][0]["result"]
dataset_drifted = drift_summary["dataset_drift"]
n_drifted       = drift_summary["number_of_drifted_columns"]
n_total         = drift_summary["number_of_columns"]
drift_share     = drift_summary["share_of_drifted_columns"]

print("=" * 50)
print("DRIFT DETECTION SUMMARY")
print("=" * 50)
print(f"  Dataset drifted:   {dataset_drifted}")
print(f"  Drifted features:  {n_drifted} / {n_total}")
print(f"  Drift share:       {drift_share*100:.1f}%")
print()

# Metrics 1 onwards have per-feature results
print("Per-feature drift results:")
print(f"{'Feature':<30} {'Drifted':>8} {'p-value':>10} {'Test':>15}")
print("-" * 65)

feature_results = {}  # we'll populate this for use in later cells

for metric in results["metrics"][1:]:
    result = metric.get("result", {})

    # skip non-feature metrics (DataQuality ones)
    if "column_name" not in result:
        continue

    feat    = result["column_name"]
    drifted = result.get("drift_detected", False)
    p_value = result.get("p_value", None)
    test    = result.get("stattest_name", result.get("stattest", ""))
    flag    = "YES" if drifted else "no"
    p_str   = f"{p_value:.4f}" if p_value is not None else "N/A"

    # store for retraining trigger cell
    feature_results[feat] = {
        "drift_detected": drifted,
        "p_value":        p_value,
        "stattest_name":  test
    }

    print(f"{feat:<30} {flag:>8} {p_str:>10} {test:>15}")

print()
print(f"feature_results populated with {len(feature_results)} features")

DRIFT DETECTION SUMMARY
  Dataset drifted:   True
  Drifted features:  10 / 12
  Drift share:       83.3%

Per-feature drift results:
Feature                         Drifted    p-value            Test
-----------------------------------------------------------------
amount_to_balance_ratio              no        N/A                
newbalanceDest                       no        N/A                
amount                               no        N/A                
orig_balance_diff                    no        N/A                
oldbalanceOrg                        no        N/A                
hour_of_day                          no        N/A                
oldbalanceDest                       no        N/A                
step                                 no        N/A                
dest_balance_diff                    no        N/A                
newbalanceOrig                       no        N/A                
type_encoded                         no        N/A             

---
## Step 5 — Auto-Retrain Trigger

If more than 30% of features have drifted, Automatically the model is retrained.
This is the core of the MLOps automated retraining loop.

In [8]:
import joblib
import mlflow
import mlflow.xgboost
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score

DRIFT_THRESHOLD = 0.30  # retrain if >30% of features drifted

print(f"Drift threshold: {DRIFT_THRESHOLD*100:.0f}%")
print(f"Actual drift:    {drift_share*100:.1f}%")
print()

if drift_share >= DRIFT_THRESHOLD:
    print("DRIFT THRESHOLD EXCEEDED — triggering automatic retraining...")
    print()

    mlflow.set_tracking_uri("mlruns")
    mlflow.set_experiment("fraud-detection")

    with mlflow.start_run(run_name=f"retrain_after_drift_{datetime.now().strftime('%Y%m%d')}"):

        mlflow.log_param("trigger",       "drift_detection")
        mlflow.log_param("drift_share",   round(drift_share, 4))
        mlflow.log_param("drifted_feats", n_drifted)

        # Combine original train + new production data for retraining
        X_new = production_data[FEATURES]
        y_new = production_data["isFraud"]

        X_combined = pd.concat([X_train, X_new], ignore_index=True)
        y_combined = pd.concat([y_train, y_new], ignore_index=True)

        print(f"Combined training set: {X_combined.shape}")

        # SMOTE on combined data
        fraud_count     = y_combined.sum()
        non_fraud_count = (y_combined == 0).sum()
        target_fraud    = int(non_fraud_count * 0.1)

        smote = SMOTE(sampling_strategy={1: target_fraud}, random_state=42, k_neighbors=5)
        X_res, y_res = smote.fit_resample(X_combined, y_combined)

        # Retrain XGBoost
        xgb_params = {
            "n_estimators":     300,
            "max_depth":        6,
            "learning_rate":    0.05,
            "subsample":        0.8,
            "colsample_bytree": 0.8,
            "scale_pos_weight": int(non_fraud_count / fraud_count),
            "random_state":     42,
            "n_jobs":           -1
        }
        mlflow.log_params(xgb_params)

        print("Retraining XGBoost on combined data...")
        retrained_model = XGBClassifier(**xgb_params)
        retrained_model.fit(X_res, y_res)

        # Evaluate on test set
        X_test_eval = pd.read_csv(PROCESSED_DIR / "X_test.csv")
        y_test_eval = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

        preds = retrained_model.predict(X_test_eval)
        proba = retrained_model.predict_proba(X_test_eval)[:, 1]
        f1  = f1_score(y_test_eval, preds)
        auc = roc_auc_score(y_test_eval, proba)

        mlflow.log_metric("retrain_f1",      round(f1, 4))
        mlflow.log_metric("retrain_roc_auc", round(auc, 4))

        print(f"Retrained model — F1: {f1:.4f}  |  ROC-AUC: {auc:.4f}")

        # Save retrained model as the new best model
        mlflow.xgboost.log_model(retrained_model, name="model",
                                 registered_model_name="FraudDetectionXGB_retrained")
        joblib.dump(retrained_model, MODELS_DIR / "best_model.pkl")

        print()
        print("Retraining complete — best_model.pkl updated")
        print(f"MLflow run logged under experiment: fraud-detection")

else:
    print("Drift within acceptable limits — no retraining needed")

Drift threshold: 30%
Actual drift:    83.3%

DRIFT THRESHOLD EXCEEDED — triggering automatic retraining...

Combined training set: (200000, 12)
Retraining XGBoost on combined data...
Retrained model — F1: 0.7009  |  ROC-AUC: 0.9970


Successfully registered model 'FraudDetectionXGB_retrained'.
Created version '1' of model 'FraudDetectionXGB_retrained'.



Retraining complete — best_model.pkl updated
MLflow run logged under experiment: fraud-detection


---
## Step 6 — Save Drift Summary as JSON

This JSON is used by the monitoring dashboard to display current drift status.

In [9]:
# Save a lightweight JSON summary for the dashboard
drift_summary = {
    "timestamp":        datetime.now().isoformat(),
    "dataset_drifted":  dataset_drifted,
    "drifted_features": n_drifted,
    "total_features":   n_total,
    "drift_share":      round(drift_share, 4),
    "threshold":        DRIFT_THRESHOLD,
    "retrain_triggered": drift_share >= DRIFT_THRESHOLD,
    "feature_drift": {
        feat: info.get("drift_detected", False)
        for feat, info in feature_results.items()
    }
}

summary_path = REPORTS_DIR / "latest_drift_summary.json"
with open(summary_path, "w") as f:
    json.dump(drift_summary, f, indent=2)

print("Drift summary saved:")
print(json.dumps(drift_summary, indent=2))

Drift summary saved:
{
  "timestamp": "2026-05-29T12:49:33.163212",
  "dataset_drifted": true,
  "drifted_features": 10,
  "total_features": 12,
  "drift_share": 0.8333,
  "threshold": 0.3,
  "retrain_triggered": true,
  "feature_drift": {
    "amount_to_balance_ratio": false,
    "newbalanceDest": false,
    "amount": false,
    "orig_balance_diff": false,
    "oldbalanceOrg": false,
    "hour_of_day": false,
    "oldbalanceDest": false,
    "step": false,
    "dest_balance_diff": false,
    "newbalanceOrig": false,
    "type_encoded": false,
    "isFraud": false,
    "orig_balance_zero": false
  }
}


---
## Summary

In [10]:
print("=" * 50)
print("DRIFT MONITORING COMPLETE")
print("=" * 50)
print(f"Reference data:      {len(reference_data):,} rows (training set)")
print(f"Production batch:    {len(production_data):,} rows (simulated drift)")
print(f"Features monitored:  {n_total}")
print(f"Features drifted:    {n_drifted} ({drift_share*100:.1f}%)")
print(f"Retrain triggered:   {drift_share >= DRIFT_THRESHOLD}")
print()
print("Reports saved:")
for f in sorted(REPORTS_DIR.glob("*")):
    print(f"  {f.name}")
print()
print("Open the HTML report in your browser to see the full Evidently dashboard.")
print()
print("Next step -> 05_docker_and_deployment.ipynb")

DRIFT MONITORING COMPLETE
Reference data:      160,000 rows (training set)
Production batch:    40,000 rows (simulated drift)
Features monitored:  12
Features drifted:    10 (83.3%)
Retrain triggered:   True

Reports saved:
  drift_report_20260529_123957.html
  latest_drift_summary.json

Open the HTML report in your browser to see the full Evidently dashboard.

Next step -> 05_docker_and_deployment.ipynb


---
---